In [ ]:
from collections import Counter
from copy import deepcopy

import nibabel as nib
import numpy as np
import torch
from skimage import metrics
from torch.nn.parallel import DataParallel as DP
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from cfg import ModelCfg, TrainCfg
from data_reader import MBAS2024
from dataset import MBAS2024Dataset
from models import Ensemble_MBAS2024, MultiHead_MBAS2024
from pipeline import run_pipeline
from trainer import MBAS2024Trainer
from utils.scoring_metrics import _hausdorff_distance, compute_challenge_metrics, hausdorff_distance

%load_ext autoreload
%autoreload 2

In [ ]:
db_dir = "/home/wenh06/Jupyter/wenhao/Hot-Data/MBAS2024/"

In [ ]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# Stage 0

In [ ]:
train_config = deepcopy(TrainCfg)
train_config.db_dir = db_dir
train_config.debug = False
train_config.stage = 0
train_config.n_epochs = 100
train_config.apply_mclahe = True
train_config.batch_size = 1

In [ ]:
model_config = deepcopy(ModelCfg)
model_config.stage = train_config.stage
model_config.seg_model_name = "vnet"  # "vnet", "nestedvnet"
model_config.apply_mclahe = train_config.apply_mclahe
model_config.seg_loss = "ExpLog_loss"  # "vnet", "nestedvnet"
model_config.seg_loss_kw = None  # "vnet", "nestedvnet"
model = MultiHead_MBAS2024(config=model_config)

In [ ]:
model

In [ ]:
# if torch.cuda.device_count() > 1:
#     model = DP(model)
# model = DDP(model)
model = model.to("cuda")

In [ ]:
if isinstance(model, DP):
    print(model.module.module_size, model.module.module_size_)
else:
    print(model.module_size, model.module_size_)

In [ ]:
trainer = MBAS2024Trainer(
    model=model,
    model_config=model_config,
    train_config=train_config,
    device=device,
)

In [ ]:
best_model_state_dict = trainer.train()

In [ ]:
trainer.log_manager.flush()
trainer.log_manager.close()

In [ ]:
del trainer, model, best_model_state_dict
torch.cuda.empty_cache()

# Stage 1

In [ ]:
train_config = deepcopy(TrainCfg)
train_config.db_dir = db_dir
train_config.debug = False
train_config.stage = 1
train_config.n_epochs = 70
train_config.apply_mclahe = True
train_config.batch_size = 1

In [ ]:
model_config = deepcopy(ModelCfg)
model_config.stage = train_config.stage
model_config.seg_model_name = "nestedvnet"  # "nestedvnet", "nestedvnet"
model_config.apply_mclahe = train_config.apply_mclahe
model_config.seg_loss = "AsymmetricLoss"
model_config.seg_loss_kw = None
model = MultiHead_MBAS2024(config=model_config)

In [ ]:
model = MultiHead_MBAS2024.from_checkpoint(
    "./saved_models/BestModel_nestedvnet-Stage1_70_05-20_15-53.pth.tar", weights_only=False, device=device
)[0]

In [ ]:
model = model.to(device=device)

In [ ]:
model.segmentation_loss

In [ ]:
if isinstance(model, DP):
    print(model.module.module_size, model.module.module_size_)
else:
    print(model.module_size, model.module_size_)

In [ ]:
trainer = MBAS2024Trainer(
    model=model,
    model_config=model_config,
    train_config=train_config,
    device=device,
)

In [ ]:
# best_model_state_dict = trainer.train()

In [ ]:
trainer.model.segmentation_loss

In [ ]:
trainer.evaluate(trainer.train_loader)

In [ ]:
trainer.log_manager.flush()
trainer.log_manager.close()

In [ ]:
del trainer, model, best_model_state_dict
torch.cuda.empty_cache()

## Ensemble model

In [ ]:
# "./saved_models/BestModel_vnet-Stage1_70_05-13_20-19.pth.tar"
# AsymmetricLoss
# {'left & right atrial walls-Dice': 0.7197343870451277,
#  'left & right atrial walls-HD95': 2.365103533751332,
#  'left & right atrial walls-IoU': 0.5634854955025754,
#  'right atrium-Dice': 0.9490513300697819,
#  'right atrium-HD95': 2.194113085129545,
#  'right atrium-IoU': 0.9033645937458387,
#  'left atrium-Dice': 0.952041875842788,
#  'left atrium-HD95': 2.2434061620575516,
#  'left atrium-IoU': 0.9085826187227863}

# "./saved_models/BestModel_vnet-Stage1_70_05-14_18-10.pth.tar"
# ExpLog_loss
# {'left & right atrial walls-Dice': 0.7076207398769584,
#  'left & right atrial walls-HD95': 2.614972984089566,
#  'left & right atrial walls-IoU': 0.5519921455461434,
#  'right atrium-Dice': 0.9419505846113015,
#  'right atrium-HD95': 2.5507633401252123,
#  'right atrium-IoU': 0.8915391106416907,
#  'left atrium-Dice': 0.947484240484136,
#  'left atrium-HD95': 2.4601662640864874,
#  'left atrium-IoU': 0.901380249270284}

vnet_model_1 = MultiHead_MBAS2024.from_checkpoint(
    "./saved_models/BestModel_vnet-Stage1_70_05-14_18-10.pth.tar", device=device, weights_only=False
)[0]
vnet_model_1 = vnet_model_1.to(device)

In [ ]:
# "./saved_models/BestModel_vnet-Stage1_70_05-14_18-10.pth.tar"
# ExpLog_loss
# {'left & right atrial walls-Dice': 0.7076207398769584,
#  'left & right atrial walls-HD95': 2.614972984089566,
#  'left & right atrial walls-IoU': 0.5519921455461434,
#  'right atrium-Dice': 0.9419505846113015,
#  'right atrium-HD95': 2.5507633401252123,
#  'right atrium-IoU': 0.8915391106416907,
#  'left atrium-Dice': 0.947484240484136,
#  'left atrium-HD95': 2.4601662640864874,
#  'left atrium-IoU': 0.901380249270284}

# "./saved_models/BestModel_nestedvnet-Stage1_70_05-20_16-33.pth.tar"
# ExpLog_loss
# 'left & right atrial walls-Dice': 0.8293850841052204,
# 'left & right atrial walls-HD95': 1.7248418655888749,
# 'left & right atrial walls-IoU': 0.7188384812777602,
# 'right atrium-Dice': 0.9447901084367325,
# 'right atrium-HD95': 2.489416791233532,
# 'right atrium-IoU': 0.9071715443894043,
# 'left atrium-Dice': 0.9604856538879393,
# 'left atrium-HD95': 1.6969297869246613,
# 'left atrium-IoU': 0.9287938060174739

vnet_model_2 = MultiHead_MBAS2024.from_checkpoint(
    "./saved_models/BestModel_nestedvnet-Stage1_70_05-20_16-33.pth.tar", device=device, weights_only=False
)[0]
vnet_model_2 = vnet_model_2.to(device)

In [ ]:
train_config = deepcopy(TrainCfg)
train_config.db_dir = db_dir
train_config.debug = False
train_config.stage = 1
train_config.n_epochs = 35
train_config.apply_mclahe = True
train_config.batch_size = 1

In [ ]:
model_config = deepcopy(ModelCfg)
model_config.stage = train_config.stage
# model_config.seg_model_name = "vnet"  # "vnet", "nestedvnet"
model_config.apply_mclahe = train_config.apply_mclahe
model_config.seg_loss = "ExpLog_loss"  # "vnet", "nestedvnet"
model_config.seg_loss_kw = None  # "vnet", "nestedvnet"
# model = MultiHead_MBAS2024(config=model_config)

In [ ]:
ensemble_model = Ensemble_MBAS2024([vnet_model_1, vnet_model_2], model_config)

In [ ]:
ensemble_model.freeze_base_models()

In [ ]:
ensemble_model = ensemble_model.to(device)

In [ ]:
ensemble_model.segmentation_loss

In [ ]:
trainer = MBAS2024Trainer(
    model=ensemble_model,
    model_config=model_config,
    train_config=train_config,
    device=device,
)

In [ ]:
best_model_state_dict = trainer.train()

In [ ]:
trainer.evaluate(trainer.train_loader)

In [ ]:
trainer.log_manager.flush()
trainer.log_manager.close()

In [ ]:
del trainer, ensemble_model, best_model_state_dict
torch.cuda.empty_cache()

# Inference

In [ ]:
stage0_model = MultiHead_MBAS2024.from_checkpoint("./saved_models/vnet/v2/stage0-model.pth.tar")[0].to(device).eval()
stage1_model = MultiHead_MBAS2024.from_checkpoint("./saved_models/vnet/v2/stage1-model.pth.tar")[0].to(device).eval()

In [ ]:
dr = MBAS2024(db_dir)

In [ ]:
image = dr.load_data(0)
ann = dr.load_ann(0)

In [ ]:
pred_mask = run_pipeline(image, stage0_model, stage1_model)
ann = ann.astype(pred_mask.dtype)

In [ ]:
compute_challenge_metrics([ann], [pred_mask])